In [47]:
def quadratic_model(p, x):
    a, b, c = p
    return a * x**2 + b * x + c

In [7]:
def calculate_x_intercepts(a, b, c):
    import numpy as np
    discriminant = b**2 - 4*a*c
    if discriminant < 0:
        return []  # No real roots
    elif discriminant == 0:
        return [-b / (2*a)]  # One real root
    else:
        root1 = (-b + np.sqrt(discriminant)) / (2*a)
        root2 = (-b - np.sqrt(discriminant)) / (2*a)
        return [min(root1,root2)]  # Two real roots, only showing first

In [2]:
def analysis(x,y,sigma_x,sigma_y):
    import pandas as pd
    import numpy as np
    import scipy as sp
    import matplotlib.pyplot as plt
    from scipy.optimize import curve_fit
    from scipy.odr import ODR, Model, RealData
    
    # Create a Model object for the quadratic function
    model = Model(quadratic_model)
    
    # Number of Monte Carlo iterations
    n_iterations = 10000
    
    # Storage for fit parameters from each iteration (3 parameters: a, b, c)
    fit_parameters = np.zeros((n_iterations, 3))
    
    for i in range(n_iterations):
        # Generate simulated data by adding random noise based on the uncertainties
        x_simulated = x + np.random.normal(0, sigma_x)
        y_simulated = y + np.random.normal(0, sigma_y)
        
        # Create a RealData object using the simulated data
        data = RealData(x_simulated, y_simulated, sx=sigma_x, sy=sigma_y)
        
        # Setup ODR with the model and simulated data
        odr = ODR(data, model, beta0=[1.0, 0.0, 0.0])  # Initial guess for a, b, c
        
        # Run the regression
        out = odr.run()
        
        # Store the fit parameters
        fit_parameters[i, :] = out.beta
    
    # Calculate the mean and standard deviation of the fit parameters
    param_means = np.mean(fit_parameters, axis=0)
    param_stds = np.std(fit_parameters, axis=0)
    
    
    # Parameters and their uncertainties
    a, b, c = param_means
    sigma_a, sigma_b, sigma_c = param_stds
    
    # Monte Carlo simulation
    n_samples = 10000
    a_samples = np.random.normal(a, sigma_a, n_samples)
    b_samples = np.random.normal(b, sigma_b, n_samples)
    c_samples = np.random.normal(c, sigma_c, n_samples)
    
    x_intercepts = []
    
    for a_sample, b_sample, c_sample in zip(a_samples, b_samples, c_samples):
        x_intercepts.append(calculate_x_intercepts(a_sample, b_sample, c_sample))
    
    # Estimate uncertainty in the x-intercept
    x_intercept_mean = np.mean(x_intercepts)
    x_intercept_uncertainty = np.std(x_intercepts)

    return(param_means,param_stds,x_intercept_mean,x_intercept_uncertainty)
